In [1]:
# 라이브러리를 사용하지 않고  Adaboost 구현
import numpy as np

class DecisionStump:
    """ 간단한 결정 스텀프 (한 가지 특성으로 분류) """
    def __init__(self): # 생성자 정의
        self.feature_index = None
        self.threshold = None
        self.polarity = 1
        self.alpha = None

    def predict(self, X): # 예측 정의
        n_samples = X.shape[0] 
        X_column = X[:, self.feature_index]
        predictions = np.ones(n_samples)
        if self.polarity == 1:
            predictions[X_column < self.threshold] = -1
        else:
            predictions[X_column > self.threshold] = -1
        return predictions

class AdaBoost:
    def __init__(self, n_clf=5):
        self.n_clf = n_clf

    def fit(self, X, y):
        n_samples, n_features = X.shape
        # 샘플 가중치 초기화
        w = np.full(n_samples, (1 / n_samples))

        self.clfs = []

        for _ in range(self.n_clf):
            clf = DecisionStump()
            min_error = float('inf')
            
            # 모든 특성에 대해 최적의 스텀프 찾기
            for feature_i in range(n_features):
                X_column = X[:, feature_i]
                thresholds = np.unique(X_column)
                for threshold in thresholds:
                    # 예측 수행
                    p = 1
                    predictions = np.ones(n_samples)
                    predictions[X_column < threshold] = -1

                    # 오차 계산
                    error = sum(w[y != predictions])

                    # 반대 방향 예측 고려
                    if error > 0.5:
                        error = 1 - error
                        p = -1

                    # 최소 오차 및 최적의 스텀프 저장
                    if error < min_error:
                        min_error = error
                        clf.polarity = p
                        clf.threshold = threshold
                        clf.feature_index = feature_i

            # alpha 계산 (학습기의 중요도)
            EPS = 1e-10  # 로그 0 방지
            clf.alpha = 0.5 * np.log((1 - min_error) / (min_error + EPS))

            # 가중치 업데이트
            predictions = clf.predict(X)
            w *= np.exp(-clf.alpha * y * predictions)
            w /= np.sum(w)

            # 학습기 저장
            self.clfs.append(clf)

    def predict(self, X):
        clf_preds = [clf.alpha * clf.predict(X) for clf in self.clfs]
        y_pred = np.sum(clf_preds, axis=0)
        y_pred = np.sign(y_pred)
        return y_pred

# 예제 데이터 생성
X = np.array([[0, 0], [1, 1], [1, 0], [0, 1]])
y = np.array([1, 1, -1, -1])  # 클래스 라벨

# 모델 학습 및 예측
clf = AdaBoost(n_clf=5)
clf.fit(X, y)
y_pred = clf.predict(X)

print("실제 라벨:", y)
print("예측 라벨:", y_pred)

실제 라벨: [ 1  1 -1 -1]
예측 라벨: [-1. -1. -1. -1.]


### GBT 코드

In [3]:
np.random.seed(0) # 시드 정의후 예시 데이터 생성
X = np.linspace(0,10,100).reshape(-1,1) 
y = np.sin(X).ravel() + np.random.normal(0, 0.1, X.shape[0])

In [4]:
## 약한 학습기의 개수 n_estimators 
## 학습률 learning_rate

n_estimators = 100
learning_rate = 0.1

# 평균값으로 일단 초기 모델을 만들자
y_pred = np.full(y.shape, y.mean())
# 수업시간에만 특정 초기 모델을 사용하지 않고 평균으로만 진행한 것임

# 예측값을 저장할 리스트
predictions = []

# 10번 돌아갈 예정 
# 잔차가 필요하다!
# 잔차를 residual = y-y_pred 
# 모델에 목표변수에 넣어서 학습했다.
# 모델이 필요하다. 
# 모델이 어떤 것이냐에 따라 모델의 비용함수가 업데이트 된다.
# 단순한 회귀 모델을 만들어서 잔차에 대해서 회귀 -> 선형회귀를 통해서 직선의 기울기와 절편 추정
# 기울기와 절편을 계산 (least squares)

# 약한 학습기인 회귀모델로 예측한 결과 나올 것이고 - 결과로 계속 반복을 진행한다.

for m in range(n_estimators):
    # 잔차를 추가하기
    residual = y - y_pred
    # 단순 선형회귀로 학습해서 기울기랑 절편을 업데이트
    gradient =np.dot(X.T, residual)/np.sum(X**2) #기울기
    intercept = residual.mean() #절편
    
    #새로운 약한 학습기인 선형 회귀로 예측값을 계산
    y_weak_learner = gradient * X.ravel() + intercept
    
    #약한 학습기의 예측 결과를 현재 모델에 추가하면서 진행
    y_pred +=learning_rate * y_weak_learner
    
    # 예측값을 저장장해야 한다.
    predictions.append(y_pred.copy()) # 각 스텝마다 들어간다.
    
#최종 예측 결과

fin_predictions = y_pred

# 최종 출력을 통해서 실제값, 예측값, 오차를 비교해 보자!


In [10]:
#10번 estimators 진행시
print('실제 값 y:', y[:5])
print('최종 예측 값 fin_predictions:', fin_predictions[:5])
print('오차 (잔차):', y[:5] - fin_predictions[:5])
print("--------------------------------------------------------------------------------")
#100번 estimators 진행시
print('실제 값 y:', y[:5])
print('최종 예측 값 fin_predictions:', fin_predictions[:5])
print('오차 (잔차):', y[:5] - fin_predictions[:5])

실제 값 y: [0.17640523 0.14085414 0.29852265 0.52250312 0.57989241]
최종 예측 값 fin_predictions: [0.25878447 0.25722101 0.25565756 0.25409411 0.25253065]
오차 (잔차): [-0.08237923 -0.11636687  0.0428651   0.26840902  0.32736176]
--------------------------------------------------------------------------------
실제 값 y: [0.17640523 0.14085414 0.29852265 0.52250312 0.57989241]
최종 예측 값 fin_predictions: [0.25878447 0.25722101 0.25565756 0.25409411 0.25253065]
오차 (잔차): [-0.08237923 -0.11636687  0.0428651   0.26840902  0.32736176]


### 사이킷 런으로 간단히 구현하기

In [11]:
from sklearn.ensemble import GradientBoostingRegressor

reg = GradientBoostingRegressor(n_estimators = 100,
                               max_depth =4,
                               random_state=111)

In [13]:
reg.fit(X,y) # 학습

GradientBoostingRegressor(max_depth=4, random_state=111)

In [14]:
X

array([[ 0.        ],
       [ 0.1010101 ],
       [ 0.2020202 ],
       [ 0.3030303 ],
       [ 0.4040404 ],
       [ 0.50505051],
       [ 0.60606061],
       [ 0.70707071],
       [ 0.80808081],
       [ 0.90909091],
       [ 1.01010101],
       [ 1.11111111],
       [ 1.21212121],
       [ 1.31313131],
       [ 1.41414141],
       [ 1.51515152],
       [ 1.61616162],
       [ 1.71717172],
       [ 1.81818182],
       [ 1.91919192],
       [ 2.02020202],
       [ 2.12121212],
       [ 2.22222222],
       [ 2.32323232],
       [ 2.42424242],
       [ 2.52525253],
       [ 2.62626263],
       [ 2.72727273],
       [ 2.82828283],
       [ 2.92929293],
       [ 3.03030303],
       [ 3.13131313],
       [ 3.23232323],
       [ 3.33333333],
       [ 3.43434343],
       [ 3.53535354],
       [ 3.63636364],
       [ 3.73737374],
       [ 3.83838384],
       [ 3.93939394],
       [ 4.04040404],
       [ 4.14141414],
       [ 4.24242424],
       [ 4.34343434],
       [ 4.44444444],
       [ 4

In [15]:
reg.predict(np.array([0.2020202]).reshape(-1,1))

array([0.29960409])